# Column Transformer

In [50]:
import pandas as pd
import numpy as np 

In [51]:
df = pd.read_csv("https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/main/day28-column-transformer/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [52]:
df.nunique()

age          55
gender        2
fever         7
cough         2
city          4
has_covid     2
dtype: int64

In [53]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [54]:
X = df.iloc[:,:5]
y = df.iloc[:,-1]

In [55]:
from sklearn.model_selection import train_test_split 
X_train, X_test , y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=2)

## Without using trasnformer

In [56]:
# SimpleImputer 
from sklearn.impute import SimpleImputer 
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [57]:
# one hot encoding'
from sklearn.preprocessing import OneHotEncoder 
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['fever', "city"]])
X_test_gender_city = ohe.fit_transform(X_test[['fever', "city"]])

X_test_gender_city.shape

(20, 9)

In [58]:
# Ordinal encoding 
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[["Mild",'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [59]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [60]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 13)

## With column transformer

In [67]:
from sklearn.compose import ColumnTransformer 
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(sparse_output=True, drop='first', handle_unknown='ignore'), ['gender', 'city'])
    ],
    remainder='passthrough'
)

In [68]:
transformer.fit_transform(X_train).shape

(80, 7)

In [70]:
transformer.transform(X_test).shape

(20, 7)